In [45]:
from pathlib import Path
from typing import Dict, List, Tuple

import arviz as az
import numpy as np
import pandas as pd
from sklearn.metrics import accuracy_score, roc_auc_score
from sklearn.model_selection import train_test_split

import cmdstanpy
cmdstanpy.rebuild_cmdstan()
from cmdstanpy import CmdStanModel


import kagglehub
from pathlib import Path
# Download latest version
path = kagglehub.dataset_download("annavictoria/speed-dating-experiment")

print("Path to dataset files:", path)

import os

files = os.listdir(path)
print(files)


Done:  (00:52) | ██████████ | --- CmdStan v2.38.0 built ---         


Path to dataset files: /Users/nidhipad/.cache/kagglehub/datasets/annavictoria/speed-dating-experiment/versions/1
['Speed Dating Data Key.doc', 'Speed Dating Data.csv']


In [46]:
csv_path = Path(path)/ "Speed Dating Data.csv"
stan_file = Path("mpt_p3.stan")
predictor_cols = ["attr", "sinc", "intel"]
target_col = "dec"

In [47]:
def load_and_clean_data(
    csv_path: Path,
    predictors: List[str],
    target: str,
) -> pd.DataFrame:

    df = pd.read_csv(csv_path, encoding="latin1")
    needed_cols = predictors + [target]
    df = df[needed_cols].dropna().copy()

    for col in predictors + [target]:
        df[col] = pd.to_numeric(df[col], errors="coerce")

    df = df.dropna().copy()
    df[target] = df[target].astype(int)

    valid_targets = {0, 1}
    df = df[df[target].isin(valid_targets)].copy()

    return df

In [48]:
def standardize_train_test(
    x_train: pd.DataFrame,
    x_test: pd.DataFrame,
) -> Tuple[np.ndarray, np.ndarray, pd.Series, pd.Series]:

    train_means = x_train.mean(axis=0)
    train_stds = x_train.std(axis=0, ddof=0)

    train_stds = train_stds.replace(0, 1.0)

    x_train_std = ((x_train - train_means) / train_stds).to_numpy()
    x_test_std = ((x_test - train_means) / train_stds).to_numpy()

    return x_train_std, x_test_std, train_means, train_stds


In [49]:
def build_stan_data(x: np.ndarray, y: np.ndarray) -> Dict[str, object]:
    return {
        "N": x.shape[0],
        "K": x.shape[1],
        "X": x,
        "y": y.astype(int),
    }

In [50]:
def fit_bayesian_logistic_regression(
    stan_file: Path,
    stan_data: Dict[str, object],
):
    model = CmdStanModel(stan_file=str(stan_file))
    fit = model.sample(
        data=stan_data,
        chains=4,
        parallel_chains=4,
        iter_warmup=1000,
        iter_sampling=1000,
        seed=RANDOM_STATE,
        refresh=200,
    )
    return fit

In [51]:
def summarize_posterior(
    fit,
    predictor_names: List[str],
) -> pd.DataFrame:

    idata = az.from_cmdstanpy(
        posterior=fit,
        observed_data={"y": fit.stan_variable("p") * 0}
    )

    summary = az.summary(
        idata,
        var_names=["alpha", "beta"],
        hdi_prob=0.95,
    ).copy()


    rename_map = {}
    for i, name in enumerate(predictor_names):
        rename_map[f"beta[{i}]"] = f"beta_{name}"

    summary = summary.rename(index=rename_map)
    return summary

In [52]:
def extract_beta_summary(fit, predictor_names):
    summary = fit.summary()
    beta_rows = [idx for idx in summary.index if idx.startswith("beta[")]

    if len(beta_rows) != len(predictor_names):
        raise ValueError(
            f"Mismatch: Stan produced {len(beta_rows)} beta coefficients, "
            f"but predictor_names has {len(predictor_names)} entries.\n"
            f"beta rows: {beta_rows}\n"
            f"predictor_names: {predictor_names}"
        )

    rows = []
    for row_name, predictor_name in zip(beta_rows, predictor_names):
        row = {
            "parameter": predictor_name,
        }

        for col in summary.columns:
            row[col] = summary.loc[row_name, col]

        rows.append(row)

    return pd.DataFrame(rows)

In [53]:
def posterior_mean_probabilities(
    fit,
    x_test: np.ndarray,
) -> np.ndarray:

    alpha_draws = fit.stan_variable("alpha")
    beta_draws = fit.stan_variable("beta")

    linear_predictor = alpha_draws[:, None] + beta_draws @ x_test.T
    probs = 1.0 / (1.0 + np.exp(-linear_predictor))
    return probs.mean(axis=0)

In [54]:
def convergence_check(summary_df: pd.DataFrame) -> bool:
    rhat_ok = (summary_df["r_hat"] < 1.01).all()
    ess_bulk_ok = (summary_df["ess_bulk"] > 400).all()
    ess_tail_ok = (summary_df["ess_tail"] > 400).all()
    return bool(rhat_ok and ess_bulk_ok and ess_tail_ok)

In [56]:
def strongest_predictor(summary_df: pd.DataFrame) -> str:
    """
    use largest absolute posterior mean among the 3 predictors to determine strongest predictor
    """
    coef_df = summary_df[summary_df["parameter"] != "alpha"].copy()
    coef_df["abs_mean"] = coef_df["mean"].abs()
    best_row = coef_df.sort_values("abs_mean", ascending=False).iloc[0]
    return str(best_row["parameter"])

In [59]:
def main() -> None:

    df = load_and_clean_data(
        csv_path=csv_path,
        predictors=predictor_cols,
        target=target_col,
    )

    x = df[predictor_cols]
    y = df[target_col].to_numpy()

    x_train, x_test, y_train, y_test = train_test_split(
        x,
        y,
        test_size=0.20,
        random_state=42,
        stratify=y,
    )

    x_train_std, x_test_std, train_means, train_stds = standardize_train_test(
        x_train=x_train,
        x_test=x_test,
    )

    stan_data = build_stan_data(x=x_train_std, y=y_train)
    fit = fit_bayesian_logistic_regression(
        stan_file=stan_file,
        stan_data=stan_data,
    )

    coef_summary = extract_beta_summary(
        fit=fit,
        predictor_names=PREDICTOR_COLUMNS,
    )

    coef_summary = coef_summary.rename(
        columns={
            "Mean": "mean",
            "StdDev": "sd",
            "R_hat": "r_hat",
            "ESS_bulk": "ess_bulk",
            "ESS_tail": "ess_tail",
        }
    )

    y_prob = posterior_mean_probabilities(fit=fit, x_test=x_test_std)
    y_pred = (y_prob >= 0.5).astype(int)

    test_accuracy = accuracy_score(y_test, y_pred)
    test_auc = roc_auc_score(y_test, y_prob)

    did_converge = convergence_check(coef_summary)
    best_predictor = strongest_predictor(coef_summary)

    print("\nSelected predictors:")
    print(predictor_cols)

    print("\nTraining standardization means:")
    print(train_means)

    print("\nTraining standardization standard deviations:")
    print(train_stds)

    print("\nPosterior summary:")
    print(coef_summary.to_string(index=False))

    print("\nConvergence result:")
    print(f"Model converged: {did_converge}")

    print("\nStrongest predictor of a 'Yes' decision:")
    print(best_predictor)

    print("\nTest-set performance:")
    print(f"Accuracy: {test_accuracy:.4f}")
    print(f"ROC AUC:  {test_auc:.4f}")


In [60]:
if __name__ == "__main__":
    main()

11:41:51 - cmdstanpy - INFO - compiling stan file /Users/nidhipad/Dropbox/Mac/Downloads/Cognitive-Modeling-HW4/mpt_p3.stan to exe file /Users/nidhipad/Dropbox/Mac/Downloads/Cognitive-Modeling-HW4/mpt_p3
11:41:56 - cmdstanpy - INFO - compiled model executable: /Users/nidhipad/Dropbox/Mac/Downloads/Cognitive-Modeling-HW4/mpt_p3
11:41:57 - cmdstanpy - INFO - CmdStan start processing

chain 1:   0%|          | 0/2000 [00:00<?, ?it/s, (Warmup)]

chain 2:   0%|          | 0/2000 [00:00<?, ?it/s, (Warmup)]


chain 3:   0%|          | 0/2000 [00:00<?, ?it/s, (Warmup)]



chain 1:  10%|█         | 200/2000 [00:00<00:02, 644.86it/s, (Warmup)]



chain 4:  10%|█         | 200/2000 [00:00<00:02, 642.28it/s, (Warmup)]

chain 2:  10%|█         | 200/2000 [00:00<00:03, 528.39it/s, (Warmup)]


chain 1:  20%|██        | 400/2000 [00:00<00:02, 629.67it/s, (Warmup)]



chain 4:  20%|██        | 400/2000 [00:00<00:02, 561.00it/s, (Warmup)]


chain 3:  20%|██        | 400/2000 [00:00<00:02, 558.43it/s, (Wa


11:42:04 - cmdstanpy - INFO - CmdStan done processing.




Selected predictors:
['attr', 'sinc', 'intel']

Training standardization means:
attr     6.220664
sinc     7.206566
intel    7.386697
dtype: float64

Training standardization standard deviations:
attr     1.945831
sinc     1.736823
intel    1.546757
dtype: float64

Posterior summary:
parameter     mean     MCSE       sd      MAD        5%      50%      95%  ess_bulk  ess_tail  ESS_bulk/s   r_hat
     attr 1.292250 0.000673 0.041034 0.041126  1.225980 1.292360 1.359470   3719.87   2589.87     163.009 1.00064
     sinc 0.008451 0.000712 0.041243 0.041513 -0.060728 0.008746 0.075205   3362.44   2899.10     147.346 1.00063
    intel 0.113435 0.000707 0.040974 0.040823  0.044300 0.114532 0.179784   3355.81   2813.70     147.056 1.00048

Convergence result:
Model converged: True

Strongest predictor of a 'Yes' decision:
attr

Test-set performance:
Accuracy: 0.7288
ROC AUC:  0.7912
